# Earth Observation case study: water and wetland signals from Sentinel-2

This notebook is a compact portfolio example for optical remote sensing and environmental data science.

**Goal:** use Sentinel-2 Level-2A optical imagery around **Ekoln / southern Uppsala, Sweden** to:

1. query satellite scenes through a STAC API,
2. select a low-cloud scene,
3. build a cloud/shadow mask,
4. derive common spectral indices for water, vegetation and chlorophyll-related optical variability,
5. create simple water / wetland-candidate masks,
6. perform a lightweight validation sanity check,
7. discuss scientific limitations and how the workflow could be made operational.

> This is a learning case study, not a production wetland map or calibrated water-quality product.


## Why this is relevant

Earth-observation services often transform multispectral reflectance into **biogeophysical or environmental indicators**.  
For wetlands and inland waters, useful first-step indices include:

- **NDWI** = (Green − NIR) / (Green + NIR)
- **MNDWI** = (Green − SWIR) / (Green + SWIR)
- **NDVI** = (NIR − Red) / (NIR + Red)
- **NDCI** = (Red Edge − Red) / (Red Edge + Red)

NDCI is often used as a chlorophyll-related optical indicator in productive/turbid waters. In this notebook it is treated only as a **relative proxy**, not as a chlorophyll-a concentration.


In [ ]:
# If needed, install:
# %pip install pystac-client planetary-computer stackstac xarray rasterio matplotlib numpy

import numpy as np
import matplotlib.pyplot as plt
import pystac_client
import planetary_computer
import stackstac


## 1. Search Sentinel-2 scenes

The Microsoft Planetary Computer exposes a public STAC API.  
We search Sentinel-2 L2A imagery for a small area covering Ekoln and southern Uppsala during summer 2025, then choose the least cloudy scene.


In [ ]:
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

# Ekoln / southern Uppsala, Sweden
bbox = [17.45, 59.60, 17.80, 59.78]

search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=bbox,
    datetime="2025-06-01/2025-08-31",
    query={"eo:cloud_cover": {"lt": 20}},
)

items = list(search.item_collection())
print(f"Found {len(items)} scenes")

if not items:
    raise RuntimeError("No scenes found. Try a wider date range or a higher cloud threshold.")

item = min(items, key=lambda x: x.properties.get("eo:cloud_cover", 100))
print("Selected:", item.id)
print("Date:", item.datetime)
print("Scene cloud cover (%):", item.properties.get("eo:cloud_cover"))


## 2. Load optical bands

We harmonize all selected bands to **20 m** resolution for a compact workflow.

Bands used:

- B02: blue
- B03: green
- B04: red
- B05: red edge
- B08: near infrared
- B11: short-wave infrared
- SCL: Sentinel-2 scene classification layer


In [ ]:
bands = ["B02", "B03", "B04", "B05", "B08", "B11", "SCL"]

cube = stackstac.stack(
    [item],
    assets=bands,
    bounds_latlon=bbox,
    resolution=20,
    chunksize=1024,
)

# Remove time dimension (one selected scene)
cube = cube.squeeze("time")

# Sentinel-2 surface reflectance bands are stored as scaled integers.
refl = cube.sel(band=["B02", "B03", "B04", "B05", "B08", "B11"]).astype("float32") / 10000.0
scl = cube.sel(band="SCL")
refl


## 3. Mask clouds and shadows

The SCL layer labels clouds, cloud shadows, cirrus and snow/ice.  
We mask these before calculating the indices.


In [ ]:
# SCL classes to mask:
# 3 = cloud shadow, 8 = medium-probability cloud,
# 9 = high-probability cloud, 10 = cirrus, 11 = snow/ice
bad_scl = [3, 8, 9, 10, 11]
valid = ~scl.isin(bad_scl)

def band(name):
    return refl.sel(band=name).where(valid)

blue  = band("B02")
green = band("B03")
red   = band("B04")
rededge = band("B05")
nir   = band("B08")
swir  = band("B11")


## 4. True-colour image


In [ ]:
rgb = np.stack([red.values, green.values, blue.values], axis=-1)
rgb = np.clip(rgb * 3.0, 0, 1)  # simple display stretch

plt.figure(figsize=(9, 7))
plt.imshow(rgb)
plt.title("Sentinel-2 true colour — Ekoln / southern Uppsala")
plt.axis("off")
plt.show()


## 5. Spectral indices

These indices compress physically meaningful spectral contrasts into simple features.


In [ ]:
eps = 1e-6

ndwi  = (green - nir) / (green + nir + eps)
mndwi = (green - swir) / (green + swir + eps)
ndvi  = (nir - red) / (nir + red + eps)
ndci  = (rededge - red) / (rededge + red + eps)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for ax, arr, title, cmap in [
    (axes[0,0], ndwi,  "NDWI — water signal", "BrBG"),
    (axes[0,1], mndwi, "MNDWI — water / wetness signal", "BrBG"),
    (axes[1,0], ndvi,  "NDVI — vegetation signal", "RdYlGn"),
    (axes[1,1], ndci,  "NDCI — chlorophyll-related optical proxy", "viridis"),
]:
    im = ax.imshow(arr.values, vmin=-1, vmax=1, cmap=cmap)
    ax.set_title(title)
    ax.axis("off")
    fig.colorbar(im, ax=ax, fraction=0.046)

plt.tight_layout()
plt.show()


## 6. Simple water and wetland-candidate mapping

This is intentionally a **transparent baseline**, not a final classifier.

- Open water: positive MNDWI and low NDVI.
- Wetland candidate: moderate-to-high vegetation signal together with a non-dry MNDWI response.

Thresholds are illustrative and should be calibrated for a real service using labelled reference data, seasonality, and local ecology.


In [ ]:
water = (mndwi > 0.0) & (ndvi < 0.20)

wetland_candidate = (
    (mndwi > -0.20) &
    (ndvi >= 0.20) &
    (ndvi < 0.75) &
    (~water)
)

classified = np.zeros(water.shape, dtype=np.uint8)
classified[water.values == 1] = 1
classified[wetland_candidate.values == 1] = 2

plt.figure(figsize=(9, 7))
plt.imshow(classified, vmin=0, vmax=2, cmap="viridis")
plt.title("Baseline classification: 0 other, 1 water, 2 wetland candidate")
plt.axis("off")
plt.show()

valid_pixels = np.isfinite(ndvi.values)
print("Water fraction of valid pixels:", np.mean(water.values[valid_pixels]))
print("Wetland-candidate fraction of valid pixels:", np.mean(wetland_candidate.values[valid_pixels]))


## 7. Water-quality exploration with NDCI

NDCI can respond to chlorophyll-related spectral changes in optically complex water.  
Here we only inspect its **spatial distribution over pixels classified as water**.

A real water-quality product would require field observations, atmospheric/adjacency-effect checks, uncertainty analysis, seasonal validation and often a locally calibrated model.


In [ ]:
ndci_water = ndci.where(water)

plt.figure(figsize=(9, 7))
im = plt.imshow(ndci_water.values, cmap="viridis")
plt.title("NDCI over detected water pixels")
plt.axis("off")
plt.colorbar(im, label="NDCI")
plt.show()

vals = ndci_water.values[np.isfinite(ndci_water.values)]
if len(vals):
    print("NDCI water-pixel summary")
    print("  n:", len(vals))
    print("  median:", float(np.nanmedian(vals)))
    print("  10th–90th percentile:", np.nanpercentile(vals, [10, 90]))


## 8. Validation sanity check

Sentinel-2's SCL contains a **water class (6)**. It is not independent ground truth because it is derived from the same satellite product, but it is useful for a first sanity check.

For a serious project, replace or complement this with:
- field observations,
- manually labelled imagery,
- national wetland / land-cover reference data,
- water-quality sampling stations,
- temporal hold-out validation across seasons and years.


In [ ]:
reference_water = (scl == 6) & valid

pred = water.values.astype(bool)
ref = reference_water.values.astype(bool)
mask = np.isfinite(ndwi.values)

tp = np.sum(pred[mask] & ref[mask])
fp = np.sum(pred[mask] & ~ref[mask])
fn = np.sum(~pred[mask] & ref[mask])

precision = tp / (tp + fp + 1e-9)
recall = tp / (tp + fn + 1e-9)
iou = tp / (tp + fp + fn + 1e-9)

print(f"Precision vs SCL-water: {precision:.3f}")
print(f"Recall vs SCL-water:    {recall:.3f}")
print(f"IoU vs SCL-water:       {iou:.3f}")


## 9. From notebook to an operational service

A production Earth-observation workflow would extend this example by:

1. **Automating acquisition** — recurring STAC queries and scene selection.
2. **Improving cloud handling** — pixel-level cloud probability and temporal compositing.
3. **Adding reference data** — field samples and authoritative wetland/land-cover layers.
4. **Validating across space and time** — multiple lakes, seasons and years.
5. **Quantifying uncertainty** — sensor noise, atmospheric effects, threshold/model uncertainty.
6. **Scaling processing** — Dask/xarray, tiling, caching and cloud execution.
7. **Packaging reusable Python modules** — separate acquisition, preprocessing, indices, classification, validation and reporting.
8. **Monitoring outputs** — quality checks, metadata, provenance and reproducible reports.

That progression mirrors the path from **exploratory analysis → validated method → operational Earth-observation service**.


## References / data source

- Sentinel-2 Level-2A imagery accessed through the **Microsoft Planetary Computer STAC API**.
- The NDCI formulation uses Sentinel-2 red (B04) and red-edge (B05) bands and is included here as a relative optical indicator only.

This notebook is intended as a compact learning and portfolio case study.
